# Conjecture 15 (Laplace K=2) Rank-Structure Notebook

This notebook tests the thesis hypothesis that the Laplace `K=2` curvature field
\(H_t(x) = -\nabla_x^2 \log p_t(x)\) is approximately rank-2, with a small
third singular component driven by perturbative coupling.

## What we test

For each configuration \((\alpha, t)\):

1. Compute \(H_t(x)\) on a 3D grid in \(x=(x_0,x_1,x_2)\).
2. Compute SVD of \(H_t(x)\), obtaining \(\sigma_1 \ge \sigma_2 \ge \sigma_3\).
3. Visualize:
   - slice heatmaps of \(\sigma_3\),
   - binned map of \(\sigma_3/\sigma_1\) in residual coordinates,
   - binned principal angle between numerical top-2 subspace and a predicted
     two-direction subspace.
4. Stability check by repeating with two finite-difference steps.

## Runtime notes

This is numerically heavy because each Hessian entry requires nested finite
differences and Fourier inversion. Use the `RUNTIME_PROFILE` switch below:

- `"quick"`: debugging / smoke test
- `"paper"`: closer to production validation

All figures are shown inline and also saved to `../figures/conjecture15/`.


In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_2d

from scores_exact import LaplaceAR1_K2

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["figure.dpi"] = 120


In [ ]:
# ------------------------------
# Configuration
# ------------------------------
RUNTIME_PROFILE = "paper"  # "quick" or "paper"

if RUNTIME_PROFILE == "quick":
    ALPHAS = [0.3, 0.8, 0.95]
    TS = [0.1, 0.5, 1.5]
    X_MAX = 2.0
    N_GRID = 5
    H_STEPS = [1e-2, 5e-3]
    FOURIER_N = 21
    FOURIER_KMAX = 8.0
elif RUNTIME_PROFILE == "paper":
    ALPHAS = [0.3, 0.8, 0.95]
    TS = [0.1, 0.5, 1.5]
    X_MAX = 5.0
    N_GRID = 17
    H_STEPS = [1e-2, 5e-3]
    FOURIER_N = 61
    FOURIER_KMAX = 12.0
else:
    raise ValueError("RUNTIME_PROFILE must be 'quick' or 'paper'.")

B = 1.0
SIGMA0 = 1.0
MU0 = 0.0

BASE_DIR = Path("../figures/conjecture15")
BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Profile: {RUNTIME_PROFILE}")
print(f"Grid: {N_GRID}^3 on [-{X_MAX}, {X_MAX}]^3")
print(f"Configs: {[(a, t) for a in ALPHAS for t in TS]}")


In [ ]:
@dataclass
class GridResult:
    alpha: float
    t: float
    h: float
    x_grid: np.ndarray
    points: np.ndarray
    svs: np.ndarray
    v1: np.ndarray
    v2: np.ndarray
    angle_deg: np.ndarray


def predicted_subspace(alpha: float) -> np.ndarray:
    """Predicted 2D subspace basis (heuristic proxy of innovation coupling directions).

    This can be updated to the exact basis from the memo if you want strict alignment.
    """
    u1 = np.array([1.0, 0.0, 0.0])
    u2 = np.array([0.0, 1.0, alpha])
    u1 = u1 / np.linalg.norm(u1)
    u2 = u2 - np.dot(u2, u1) * u1
    u2 = u2 / np.linalg.norm(u2)
    return np.stack([u1, u2], axis=1)  # shape (3, 2)


def principal_angle_deg(U_num: np.ndarray, U_pred: np.ndarray) -> float:
    """Largest principal angle between two 2D subspaces in R^3."""
    M = U_num.T @ U_pred
    svals = np.linalg.svd(M, compute_uv=False)
    svals = np.clip(svals, -1.0, 1.0)
    theta = np.arccos(np.min(svals))
    return float(np.degrees(theta))


def hessian_from_score(model, x: np.ndarray, t: float, h: float, fourier_n: int, fourier_kmax: float) -> np.ndarray:
    """H = -∇^2 log p_t from finite differences of score."""
    H = np.zeros((3, 3), dtype=float)
    for j in range(3):
        xp = x.copy(); xp[j] += h
        xm = x.copy(); xm[j] -= h
        sp = model.score(xp, t=t, h=h, N=fourier_n, K_max=fourier_kmax)
        sm = model.score(xm, t=t, h=h, N=fourier_n, K_max=fourier_kmax)
        # d/dx_j score_i = (sp_i - sm_i)/(2h) = ∂_j ∂_i log p
        H[:, j] = -(sp - sm) / (2.0 * h)
    # enforce symmetry numerically
    return 0.5 * (H + H.T)


def run_grid(alpha: float, t: float, h: float, x_max: float, n_grid: int, fourier_n: int, fourier_kmax: float) -> GridResult:
    model = LaplaceAR1_K2(alpha=alpha, b=B, mu0=MU0, sigma0=SIGMA0)
    U_pred = predicted_subspace(alpha)

    xs = np.linspace(-x_max, x_max, n_grid)
    mesh = np.meshgrid(xs, xs, xs, indexing="ij")
    points = np.stack(mesh, axis=-1).reshape(-1, 3)

    svs = np.zeros((points.shape[0], 3), dtype=float)
    v1 = np.zeros((points.shape[0], 3), dtype=float)
    v2 = np.zeros((points.shape[0], 3), dtype=float)
    angles = np.zeros(points.shape[0], dtype=float)

    total = points.shape[0]
    for idx in range(total):
        if idx % max(total // 10, 1) == 0:
            print(f"Progress alpha={alpha}, t={t}, h={h:g}: {idx}/{total}")
        x = points[idx]
        H = hessian_from_score(model, x, t=t, h=h, fourier_n=fourier_n, fourier_kmax=fourier_kmax)
        _, s, Vt = np.linalg.svd(H)
        svs[idx] = s
        v1[idx] = Vt[0]
        v2[idx] = Vt[1]
        U_num = Vt[:2].T
        angles[idx] = principal_angle_deg(U_num, U_pred)

    return GridResult(
        alpha=alpha,
        t=t,
        h=h,
        x_grid=xs,
        points=points,
        svs=svs,
        v1=v1,
        v2=v2,
        angle_deg=angles,
    )


In [ ]:
def reshape_scalar_field(result: GridResult, values: np.ndarray) -> np.ndarray:
    n = result.x_grid.size
    return values.reshape(n, n, n)


def residual_coordinates(points: np.ndarray, alpha: float):
    x0, x1, x2 = points[:, 0], points[:, 1], points[:, 2]
    r1 = x1 - alpha * x0
    r2 = x2 - alpha * x1
    return r1, r2


def plot_sigma3_slices(result: GridResult, out_dir: Path):
    sigma3 = reshape_scalar_field(result, result.svs[:, 2])
    xs = result.x_grid
    mid = len(xs) // 2

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

    im0 = axes[0].imshow(sigma3[mid, :, :], origin="lower", extent=[xs[0], xs[-1], xs[0], xs[-1]], aspect="auto")
    axes[0].set_title("sigma3 slice at x0=0")
    axes[0].set_xlabel("x1")
    axes[0].set_ylabel("x2")
    fig.colorbar(im0, ax=axes[0], shrink=0.9)

    im1 = axes[1].imshow(sigma3[:, mid, :], origin="lower", extent=[xs[0], xs[-1], xs[0], xs[-1]], aspect="auto")
    axes[1].set_title("sigma3 slice at x1=0")
    axes[1].set_xlabel("x0")
    axes[1].set_ylabel("x2")
    fig.colorbar(im1, ax=axes[1], shrink=0.9)

    im2 = axes[2].imshow(sigma3[:, :, mid], origin="lower", extent=[xs[0], xs[-1], xs[0], xs[-1]], aspect="auto")
    axes[2].set_title("sigma3 slice at x2=0")
    axes[2].set_xlabel("x0")
    axes[2].set_ylabel("x1")
    fig.colorbar(im2, ax=axes[2], shrink=0.9)

    fig.suptitle(f"Laplace K=2: sigma3 slices (alpha={result.alpha}, t={result.t}, h={result.h:g})")
    path = out_dir / f"sigma3_slices_alpha{result.alpha}_t{result.t}_h{result.h:g}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return path


def plot_binned_map(result: GridResult, quantity: np.ndarray, title: str, cbar_label: str, out_name: str, bins: int = 25):
    r1, r2 = residual_coordinates(result.points, result.alpha)
    stat, x_edge, y_edge, _ = binned_statistic_2d(r1, r2, quantity, statistic="mean", bins=bins)

    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    im = ax.imshow(
        stat.T,
        origin="lower",
        extent=[x_edge[0], x_edge[-1], y_edge[0], y_edge[-1]],
        aspect="auto",
    )
    ax.set_xlabel("r1 = x1 - alpha*x0")
    ax.set_ylabel("r2 = x2 - alpha*x1")
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)

    out_path = out_dir / out_name
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return out_path


In [ ]:
results = {}

for alpha in ALPHAS:
    for t in TS:
        for h in H_STEPS:
            res = run_grid(
                alpha=alpha,
                t=t,
                h=h,
                x_max=X_MAX,
                n_grid=N_GRID,
                fourier_n=FOURIER_N,
                fourier_kmax=FOURIER_KMAX,
            )
            key = (alpha, t, h)
            results[key] = res

print(f"Computed {len(results)} grid results.")


In [ ]:
def summarize_result(res: GridResult) -> dict:
    ratio = res.svs[:, 2] / np.maximum(res.svs[:, 0], 1e-12)
    return {
        "sigma3_mean": float(np.mean(res.svs[:, 2])),
        "sigma3_median": float(np.median(res.svs[:, 2])),
        "sigma3_max": float(np.max(res.svs[:, 2])),
        "ratio_mean": float(np.mean(ratio)),
        "ratio_median": float(np.median(ratio)),
        "ratio_q95": float(np.quantile(ratio, 0.95)),
        "angle_mean_deg": float(np.mean(res.angle_deg)),
        "angle_q95_deg": float(np.quantile(res.angle_deg, 0.95)),
    }


rows = []
for alpha in ALPHAS:
    for t in TS:
        for h in H_STEPS:
            res = results[(alpha, t, h)]
            out_dir = BASE_DIR / f"alpha_{alpha}_t_{t}"
            out_dir.mkdir(parents=True, exist_ok=True)

            p1 = plot_sigma3_slices(res, out_dir)

            ratio = res.svs[:, 2] / np.maximum(res.svs[:, 0], 1e-12)
            p2 = plot_binned_map(
                res,
                quantity=ratio,
                title=f"sigma3/sigma1 in residual coordinates (alpha={alpha}, t={t}, h={h:g})",
                cbar_label="mean sigma3/sigma1",
                out_name=f"ratio_map_alpha{alpha}_t{t}_h{h:g}.png",
            )

            p3 = plot_binned_map(
                res,
                quantity=res.angle_deg,
                title=f"Subspace angle in residual coordinates (alpha={alpha}, t={t}, h={h:g})",
                cbar_label="mean principal angle (deg)",
                out_name=f"angle_map_alpha{alpha}_t{t}_h{h:g}.png",
            )

            s = summarize_result(res)
            s.update({"alpha": alpha, "t": t, "h": h, "sigma3_fig": str(p1), "ratio_fig": str(p2), "angle_fig": str(p3)})
            rows.append(s)

rows


In [ ]:
# Stability check across finite-difference steps

if len(H_STEPS) >= 2:
    h1, h2 = H_STEPS[0], H_STEPS[1]
    for alpha in ALPHAS:
        for t in TS:
            r1 = results[(alpha, t, h1)]
            r2 = results[(alpha, t, h2)]

            ratio1 = r1.svs[:, 2] / np.maximum(r1.svs[:, 0], 1e-12)
            ratio2 = r2.svs[:, 2] / np.maximum(r2.svs[:, 0], 1e-12)

            rel_diff = np.abs(ratio1 - ratio2) / np.maximum(np.abs(ratio1), 1e-12)
            print(
                f"alpha={alpha}, t={t} | median rel diff ratio={np.median(rel_diff):.3e}, "
                f"q95={np.quantile(rel_diff, 0.95):.3e}"
            )


## Interpretation template (fill after full sweep)

Use the printed statistics and maps to classify the result:

- **Confirmed at near-machine precision** if `sigma3/sigma1` is uniformly tiny and stable.
- **Confirmed with perturbative correction** if `sigma3/sigma1` is small but structured,
  and grows in regimes expected from coupling.
- **Refuted** if `sigma3/sigma1` is not small over broad regions or unstable to step size.

Record a one-line verdict per \((\alpha,t)\), then aggregate into a final thesis-level claim.
